In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.ensemble import ExtraTreesRegressor
import optuna
import shap
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import cross_val_score, KFold
import joblib

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
#Global font/size settings
mpl.rcParams.update({
    'font.size':          16,    # base font size
    'axes.titlesize':     20,    # subplot titles
    'axes.labelsize':     18,    # x/y axis labels
    'xtick.labelsize':    16,    # x tick labels
    'ytick.labelsize':    16,    # y tick labels
    'legend.fontsize':    16,    # legend text
    'figure.titlesize':   20,    # suptitle})

In [ ]:
ruta = r"C:\Users\Sergi\Desktop\Investigacion de materiales\Codigo definitivo\RegresionLCE_smiles\bdelectrolitos.csv"
df = pd.read_csv(ruta)
print(f"Dataset loaded: {df.shape[0]} records, {df.shape[1]} columns")
df['logCE'] = -np.log10(1 - df['CE']/100)
print(f"\nTarget statistics  logCE:")
print(df['logCE'].describe().round(4))
print(df.tail)

In [ ]:
#descriptores
descriptores = ['O', 'C', 'F', 'sO', 'sC', 'sF', 'aO', 'aC', 'aF', 'FO', 'FC', 'OC', 'InOr']
#estadísticas descriptivas
tabla_1_stats = df[descriptores].describe().T[['min', 'max', 'mean', 'std']]
print("Estadísticas para la Tabla 1")
print(tabla_1_stats.round(4))

print("rango de CE")
print(f"Valor mínimo de CE: {df['CE'].min()}%")
print(f"Valor máximo de CE: {df['CE'].max()}%")

In [ ]:
#CE distribution
fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(df['CE'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
ax.set_title('CE Distribution')
ax.set_xlabel('Coulombic Efficiency (CE)')
ax.set_ylabel('Frequency')
ax.axvline(df['CE'].mean(), color='red', linestyle='--', label=f"Mean = {df['CE'].mean():.3f}")
ax.legend()
plt.tight_layout()
plt.savefig('./resultados_CE/dist_CE.png', dpi=300)
plt.show()
#logCE distribution
fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(df['logCE'], bins=30, color='darkorange', edgecolor='white', alpha=0.85)
ax.set_title('logCE Distribution  ← Model Target')
ax.set_xlabel('logCE')
ax.set_ylabel('Frequency')
ax.axvline(df['logCE'].mean(), color='red', linestyle='--', label=f"Mean = {df['logCE'].mean():.3f}")
ax.legend()
plt.suptitle('Target Variable: Coulombic Efficiency', fontweight='bold')
plt.tight_layout()
plt.savefig('./resultados_CE/dist_logCE.png', dpi=300)
plt.show()

In [ ]:
#Target variable
y = df['logCE']
cols_excluir = ['Electrolyte', 'CE', 'LCE','logCE']
X = df.drop(columns=cols_excluir)

print(f"Features in X: {X.shape[1]} columns")
print(list(X.columns))

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
# Split 30% validation 15% and test (15%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print(f"Dataset split:")
print(f"  Train: {X_train.shape[0]} samples  ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"  Validation: {X_val.shape[0]} samples  ({X_val.shape[0]/len(X)*100:.0f}%)")
print(f"  Test: {X_test.shape[0]} samples  ({X_test.shape[0]/len(X)*100:.0f}%)")

In [ ]:
import joblib

joblib.dump({'X_train': X_train, 'y_train': y_train, 'X_val':   X_val,   'y_val':   y_val, 'X_test':  X_test,  'y_test':  y_test, 'feature_names': list(X_train.columns)}, './resultados_CE/data_splits.pkl')
print("ruta en ./resultados_CE/data_splits.pkl")

In [ ]:
#funcion de evaluacion
def calculate_metrics(y_real, y_pred):
    """Returns MSE, RMSE, MAE and R2 for a real/predicted pair."""
    return {'MSE':  mean_squared_error(y_real, y_pred), 'RMSE': np.sqrt(mean_squared_error(y_real, y_pred)),
        'MAE':  mean_absolute_error(y_real, y_pred), 'R2':   r2_score(y_real, y_pred)
    }

def evaluate_model(name, model, X_tr, y_tr, X_val, y_val, X_te, y_te, folder='./resultados_CE'):

    os.makedirs(folder, exist_ok=True)
    model.fit(X_tr, y_tr)
    y_pred_tr  = model.predict(X_tr)
    y_pred_val = model.predict(X_val)
    y_pred_te  = model.predict(X_te)
    m_tr  = calculate_metrics(y_tr,  y_pred_tr)
    m_val = calculate_metrics(y_val, y_pred_val)
    m_te  = calculate_metrics(y_te,  y_pred_te)


    print(f"  {name}")
    print(f"{'Metric':<10} {'Train':>12} {'Validation':>12} {'Test':>12}")
    for met in ['MSE', 'RMSE', 'MAE', 'R2']:
        print(f"{met:<10} {m_tr[met]:>12.4f} {m_val[met]:>12.4f} {m_te[met]:>12.4f}")
    n_out = np.sum(y_pred_te <= 0)
    print(f"Out-of-range predictions (Test):")
    print(f"Non-positive logCE predictions : {n_out} / {len(y_pred_te)}")
    if n_out > 0:
        print(f"WARNING: {n_out} predictions are non-positive — model extrapolating beyond training range.")
    valid_mask = y_pred_te > 0
    if valid_mask.sum() < len(y_pred_te):
        print(f"MAE (CE scale) computed on {valid_mask.sum()} valid predictions only.")
        
    ce_pred = 1 - 10**(-y_pred_te[valid_mask])
    ce_real = 1 - 10**(-y_te.values[valid_mask])
    mae_ce  = mean_absolute_error(ce_real, ce_pred)
    print(f"MAE in original CE scale (Test): {mae_ce:.5f}")

    #Plot 1a: Predicted vs Real — Train
    sets = [('Train',      y_tr,  y_pred_tr,  'steelblue',   m_tr), ('Validation', y_val, y_pred_val, 'seagreen',    m_val), ('Test',       y_te,  y_pred_te,  'darkorange',  m_te),]
    for label, y_r, y_p, color, m in sets:
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(y_r, y_p, alpha=0.7, color=color, edgecolor='white', s=60)
        lims = [min(float(y_r.min()), float(y_p.min())) - 0.2,
                max(float(y_r.max()), float(y_p.max())) + 0.2]
        ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
        ax.set_xlabel('Real  logCE')
        ax.set_ylabel('Predicted  logCE')
        ax.set_title(f'{label}  |  R²={m["R2"]:.3f}  RMSE={m["RMSE"]:.3f}')
        ax.legend()
        plt.suptitle(f'Predicted vs Real — {name} — {label}', fontweight='bold')
        plt.tight_layout()
        slug = name.lower().replace(" ", "_")
        plt.savefig(f'{folder}/pred_vs_real_{slug}_{label.lower()}.png', dpi=300)
        plt.show()

    #Plot 2a: Residuals vs Predicted (Test)
    residuals = y_te.values - y_pred_te
    slug = name.lower().replace(" ", "_")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(y_pred_te, residuals, alpha=0.7, color='darkorange',
               edgecolor='white', s=60)
    ax.axhline(0, color='red', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Predicted  logCE')
    ax.set_ylabel('Residual (Real - Predicted)')
    ax.set_title(f'Residuals vs Predicted (Test) — {name}')
    plt.tight_layout()
    plt.savefig(f'{folder}/residuals_vs_pred_{slug}.png', dpi=300)
    plt.show()

    #Plot 2b: Residual distribution (Test)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.hist(residuals, bins=20, color='darkorange', edgecolor='white', alpha=0.85)
    ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Residual')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Residual Distribution (Test) — {name}')
    plt.tight_layout()
    plt.savefig(f'{folder}/residuals_dist_{slug}.png', dpi=300)
    plt.show()

    #Plot 3: Metrics per split
    splits  = ['Train', 'Validation', 'Test']
    colors  = ['steelblue', 'seagreen', 'darkorange']

    for met in ['MSE', 'RMSE', 'MAE', 'R2']:
        values = [m_tr[met], m_val[met], m_te[met]]
        fig, ax = plt.subplots(figsize=(7, 6))
        bars = ax.bar(splits, values, color=colors, edgecolor='white', width=0.5)
        ax.set_title(f'{met} by Split — {name}')
        ax.set_ylabel(met)
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.001,
                    f'{val:.4f}', ha='center', va='bottom', fontsize=13)
        plt.tight_layout()
        plt.savefig(f'{folder}/metric_{met.lower()}_{slug}.png', dpi=300)
        plt.show()

    safe_name = name.lower().replace(" ", "_")
    joblib.dump(model, f'{folder}/model_{safe_name}.pkl')
    print(f"Modelo guardado: {folder}/model_{safe_name}.pkl")
    return model, {
        'MSE_tr':  m_tr['MSE'],  'RMSE_tr':  m_tr['RMSE'],  'MAE_tr':  m_tr['MAE'],  'R2_tr':  m_tr['R2'],
        'MSE_val': m_val['MSE'], 'RMSE_val': m_val['RMSE'], 'MAE_val': m_val['MAE'], 'R2_val': m_val['R2'],
        'MSE_te':  m_te['MSE'],  'RMSE_te':  m_te['RMSE'],  'MAE_te':  m_te['MAE'],  'R2_te':  m_te['R2'],
    }

In [ ]:
from sklearn.utils import resample
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def get_bootstrap_metrics(y_true, y_pred, n_iterations=1000, ci=95):
    metrics = {'r2': [], 'mae': [], 'rmse': []}
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    for _ in range(n_iterations):
        indices = resample(np.arange(len(y_true)))
        y_true_bs = y_true[indices]
        y_pred_bs = y_pred[indices]
        metrics['r2'].append(r2_score(y_true_bs, y_pred_bs))
        metrics['mae'].append(mean_absolute_error(y_true_bs, y_pred_bs))
        metrics['rmse'].append(np.sqrt(mean_squared_error(y_true_bs, y_pred_bs)))
    lower_bound = (100 - ci) / 2
    upper_bound = 100 - lower_bound
    results = {}
    for m in metrics:
        results[m] = {'mean': np.mean(metrics[m]),'low': np.percentile(metrics[m], lower_bound), 'high': np.percentile(metrics[m], upper_bound)}
    return results

In [ ]:
#XGBOOST OPTUNA
def objective_xgb(trial):
    params = {'n_estimators':     trial.suggest_int('n_estimators', 50, 500),
        'max_depth':        trial.suggest_int('max_depth', 2, 4),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, .9),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-5, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-5, 10.0, log=True),
        'objective':        'reg:squarederror',
        'random_state':     42,
        'n_jobs':           -1}
    model = XGBRegressor(**params)
    model.fit(X_train, y_train)
    y_pred_val = model.predict(X_val)
    return -np.sqrt(mean_squared_error(y_val, y_pred_val))
study_xgb = optuna.create_study(direction='maximize', study_name='XGBoost_CE')
study_xgb.optimize(objective_xgb, n_trials=400)

print("\nBest hyperparameters (XGBoost):")
for k, v in study_xgb.best_params.items():
    print(f"  {k}: {v}")

best_params_xgb = {**study_xgb.best_params, 'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1}
pipeline_xgb = Pipeline([('xgb', XGBRegressor(**best_params_xgb))])
model_xgb, metrics_xgb = evaluate_model('XGBoost', pipeline_xgb, X_train, y_train, X_val, y_val, X_test, y_test)

In [ ]:
from sklearn.metrics import r2_score
from sklearn.utils import resample
import numpy as np
y_pred_test = model_xgb.predict(X_test)
n_iterations = 1000
bootstrapped_r2 = []
for i in range(n_iterations):
    indices = resample(range(len(y_test)), replace=True, random_state=i)
    score = r2_score(y_test.iloc[indices], y_pred_test[indices])
    bootstrapped_r2.append(score)

#promedio y los límites del 95% de confianza
mean_r2 = np.mean(bootstrapped_r2)
lower_ci = np.percentile(bootstrapped_r2, 2.5)   # Límite inferior
upper_ci = np.percentile(bootstrapped_r2, 97.5)  # Límite superior

print(f"Mean R²: {mean_r2:.4f}")
print(f"95% Confidence Interval: [{lower_ci:.4f}, {upper_ci:.4f}]")

In [ ]:
#EXTRA TREES WITH OPTUNA
def objective_et(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 500),
        'max_depth':         trial.suggest_int('max_depth', 4, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'random_state':      42,
        'n_jobs':            -1
    }
    model = ExtraTreesRegressor(**params)
    model.fit(X_train, y_train)
    y_pred_val = model.predict(X_val)
    return -np.sqrt(mean_squared_error(y_val, y_pred_val))

study_et = optuna.create_study(direction='maximize', study_name='ExtraTrees_CE')
study_et.optimize(objective_et, n_trials=400)
print("\nBest hyperparameters (Extra Trees):")
for k, v in study_et.best_params.items():
    print(f"  {k}: {v}")
best_params_et = {**study_et.best_params, 'random_state': 42, 'n_jobs': -1}

pipeline_et = Pipeline([('et', ExtraTreesRegressor(**best_params_et))])
model_et, metrics_et = evaluate_model('Extra Trees', pipeline_et, X_train, y_train, X_val, y_val, X_test, y_tes)

In [ ]:
from sklearn.metrics import r2_score
from sklearn.utils import resample
import numpy as np
y_pred_test = model_et.predict(X_test)
n_iterations = 1000
bootstrapped_r2 = []

for i in range(n_iterations):
    indices = resample(range(len(y_test)), replace=True, random_state=i)
    score = r2_score(y_test.iloc[indices], y_pred_test[indices])
    bootstrapped_r2.append(score)
mean_r2 = np.mean(bootstrapped_r2)
lower_ci = np.percentile(bootstrapped_r2, 2.5)  
upper_ci = np.percentile(bootstrapped_r2, 97.5)
print(f"Mean R²: {mean_r2:.4f}")
print(f"95% Confidence Interval: [{lower_ci:.4f}, {upper_ci:.4f}]")

In [ ]:
#LIGHTGBM OPTUNA

def objective_lgbm(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 50, 500),
        'max_depth':        trial.suggest_int('max_depth', 4, 20),
        'learning_rate':    trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'num_leaves':       trial.suggest_int('num_leaves', 20, 150),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-5, 5.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-5, 5.0, log=True),
        'random_state':     42,
        'n_jobs':           -1,
        'verbose':          -1
    }
    model = LGBMRegressor(**params)
    model.fit(X_train, y_train)
    y_pred_val = model.predict(X_val)
    return -np.sqrt(mean_squared_error(y_val, y_pred_val))

study_lgbm = optuna.create_study(direction='maximize', study_name='LightGBM_CE')
study_lgbm.optimize(objective_lgbm, n_trials=400)

print("\nBest hyperparameters (LightGBM):")
for k, v in study_lgbm.best_params.items():
    print(f"  {k}: {v}")
print(f"\nBest Val R²: {study_lgbm.best_value:.4f}")

best_params_lgbm = {**study_lgbm.best_params, 'random_state': 42, 'n_jobs': -1, 'verbose': -1}

pipeline_lgbm = Pipeline([('lgbm', LGBMRegressor(**best_params_lgbm))])
model_lgbm, metrics_lgbm = evaluate_model('LightGBM', pipeline_lgbm, X_train, y_train, X_val, y_val, X_test, y_test)

In [ ]:
from sklearn.metrics import r2_score
from sklearn.utils import resample
import numpy as np
y_pred_test = model_lgbm.predict(X_test)
n_iterations = 1000
bootstrapped_r2 = []

for i in range(n_iterations):
    indices = resample(range(len(y_test)), replace=True, random_state=i)
    score = r2_score(y_test.iloc[indices], y_pred_test[indices])
    bootstrapped_r2.append(score)
mean_r2 = np.mean(bootstrapped_r2)
lower_ci = np.percentile(bootstrapped_r2, 2.5)   
upper_ci = np.percentile(bootstrapped_r2, 97.5)
print(f"Mean R²: {mean_r2:.4f}")
print(f"95% Confidence Interval: [{lower_ci:.4f}, {upper_ci:.4f}]")

In [ ]:
#CATBOOST OPTUNA
def objective_cat(trial):
    params = {
        'iterations':        trial.suggest_int('iterations', 50, 500),
        'depth':             trial.suggest_int('depth', 2, 9),
        'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'l2_leaf_reg':       trial.suggest_float('l2_leaf_reg', 1e-5, 10.0, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        'random_seed':       42,
        'verbose':           0
    }
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train)
    y_pred_val = model.predict(X_val)
    return -np.sqrt(mean_squared_error(y_val, y_pred_val))

study_cat = optuna.create_study(direction='maximize', study_name='CatBoost_CE')
study_cat.optimize(objective_cat, n_trials=400)

print("\nBest hyperparameters (CatBoost):")
for k, v in study_cat.best_params.items():
    print(f"  {k}: {v}")
print(f"\nBest Val R²: {study_cat.best_value:.4f}")

best_params_cat = {**study_cat.best_params, 'random_seed': 42, 'verbose': 0}
pipeline_cat = Pipeline([('cat', CatBoostRegressor(**best_params_cat))])

model_cat, metrics_cat = evaluate_model('CatBoost', pipeline_cat, X_train, y_train, X_val, y_val, X_test, y_test)

In [ ]:
from sklearn.metrics import r2_score
from sklearn.utils import resample
import numpy as np
y_pred_test = model_cat.predict(X_test)
n_iterations = 1000
bootstrapped_r2 = []
for i in range(n_iterations):
    indices = resample(range(len(y_test)), replace=True, random_state=i)
    score = r2_score(y_test.iloc[indices], y_pred_test[indices])
    bootstrapped_r2.append(score)
mean_r2 = np.mean(bootstrapped_r2)
lower_ci = np.percentile(bootstrapped_r2, 2.5)
upper_ci = np.percentile(bootstrapped_r2, 97.5)
print(f"Mean R²: {mean_r2:.4f}")
print(f"95% Confidence Interval: [{lower_ci:.4f}, {upper_ci:.4f}]")

In [ ]:
#FINAL MODEL COMPARISON
os.makedirs('./resultados_CE', exist_ok=True)

results = pd.DataFrame({'Model':    ['XGBoost', 'Extra Trees', 'LightGBM', 'CatBoost'],
    'MSE':      [metrics_xgb['MSE_te'],  metrics_et['MSE_te'],  metrics_lgbm['MSE_te'],  metrics_cat['MSE_te']],
    'RMSE':     [metrics_xgb['RMSE_te'], metrics_et['RMSE_te'], metrics_lgbm['RMSE_te'], metrics_cat['RMSE_te']],
    'MAE':      [metrics_xgb['MAE_te'],  metrics_et['MAE_te'],  metrics_lgbm['MAE_te'],  metrics_cat['MAE_te']],
    'R2':       [metrics_xgb['R2_te'],   metrics_et['R2_te'],   metrics_lgbm['R2_te'],   metrics_cat['R2_te']],})

print("  FINAL MODEL COMPARISON")
print(results.to_string(index=False))

models  = ['XGBoost', 'Extra Trees', 'LightGBM', 'CatBoost']
colors  = ['#2196F3', '#FF9800', '#4CAF50', '#9C27B0']

for metric in ['MSE', 'RMSE', 'MAE', 'R2']:
    values = results[metric].values
    fig, ax = plt.subplots(figsize=(8, 6))
    bars = ax.bar(models, values, color=colors, edgecolor='white', width=0.5)
    ax.set_title(f'Model Comparison — {metric}')
    ax.set_ylabel(metric)
    ax.set_xticklabels(models, rotation=15, ha='right')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=13)
    plt.suptitle('Model Comparison — CE Prediction', fontweight='bold')
    plt.tight_layout()
    slug = metric.lower().replace(" ", "_").replace("(", "").replace(")", "")
    plt.savefig(f'./resultados_CE/comparison_{slug}.png', dpi=300)
    plt.show()

In [ ]:
#SHAP ANALYSIS BEST MODEL
import shap
model_registry = {
    'XGBoost':     model_xgb,
    'Extra Trees': model_et,
    'LightGBM':    model_lgbm,
    'CatBoost':    model_cat,
}
best_name  = results.loc[results['RMSE'].idxmin(), 'Model']
best_model = model_registry[best_name]
print(f"Best model by Test RMSE: {best_name}")
estimator = best_model.named_steps[list(best_model.named_steps.keys())[-1]]
if best_name == 'XGBoost':
    booster = estimator.get_booster()
    config  = booster.save_config()
    import json
    cfg = json.loads(config)
    cfg['learner']['learner_model_param']['base_score'] = '0.5'
    booster.load_config(json.dumps(cfg))
    explainer   = shap.TreeExplainer(booster)
    shap_values = explainer.shap_values(X_test)
else:
    explainer   = shap.TreeExplainer(estimator)
    shap_values = explainer.shap_values(X_test)

#SHAP summary (beeswarm)
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.title(f'SHAP Summary — {best_name}', fontsize=17, fontweight='bold')
plt.tight_layout()
plt.savefig('./resultados_CE/shap_summary.png', dpi=300)
plt.show()

#SHAP feature importance (bar)
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title(f'SHAP Feature Importance — {best_name}', fontsize=17, fontweight='bold')
plt.tight_layout()
plt.savefig('./resultados_CE/shap_importance.png', dpi=300)
plt.show()

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

feat_names = X_test.columns.tolist()
mean_shap  = np.abs(shap_values).mean(axis=0)
sorted_idx = np.argsort(mean_shap)[::-1]
sv_sorted = shap_values[:, sorted_idx]
fn_sorted = [feat_names[i] for i in sorted_idx]
fv_sorted = X_test.values[:, sorted_idx]
shap_custom = LinearSegmentedColormap.from_list('shap_custom', ['#1E90FF', '#CC00CC', '#FF1493'], N=256)
fig, ax = plt.subplots(figsize=(14, 5))

for i, (sv_col, fv_col) in enumerate(zip(sv_sorted.T, fv_sorted.T)):
    fv_norm = (fv_col - fv_col.min()) / (fv_col.max() - fv_col.min() + 1e-9)
    colors  = shap_custom(fv_norm)

    np.random.seed(42)
    jitter = np.random.uniform(-0.2, 0.2, size=len(sv_col))
    order  = np.argsort(fv_norm)
    ax.scatter(np.full_like(sv_col, i)[order] + jitter[order], sv_col[order], c=colors[order], s=25, alpha=0.7, zorder=3, linewidths=0, rasterized=True)
ax.axhline(0, color='#aaaaaa', linewidth=0.9, linestyle='-', zorder=1)
for i in range(len(fn_sorted)):
    if i % 2 == 0:
        ax.axvspan(i - 0.5, i + 0.5, color='#f5f5f5', zorder=0)
ax.set_xticks(range(len(fn_sorted)))
ax.set_xticklabels(fn_sorted, rotation=45, ha='right', fontsize=18,
                   fontfamily='serif')
ax.set_ylabel('SHAP value (impact on model output)',
              fontsize=20, fontfamily='serif')
ax.set_xlim(-0.5, len(fn_sorted) - 0.5)
ax.tick_params(axis='both', which='both', length=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('#cccccc')
ax.spines['bottom'].set_color('#cccccc')
ax.yaxis.grid(True, linestyle='-', linewidth=0.5, color='#eeeeee', zorder=0)
ax.set_axisbelow(True)
sm = plt.cm.ScalarMappable(cmap=shap_custom, norm=plt.Normalize(vmin=0, vmax=1))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.015, pad=0.02, aspect=25)
cbar.set_label('Feature value', fontsize=14, fontfamily='serif')
cbar.set_ticks([0, 1])
cbar.set_ticklabels(['Low', 'High'], fontsize=13)
cbar.outline.set_visible(False)
cbar.ax.tick_params(length=0)

plt.title(f'SHAP Beeswarm — {best_name}', fontsize=30, fontweight='bold', fontfamily='serif', pad=12)
plt.tight_layout()
plt.savefig('./resultados_CE/shap_beeswarm_custom.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
#SHAP ANALYSIS
import shap
model_registry = {
    'XGBoost':     model_xgb,
    'Extra Trees': model_et,
    'LightGBM':    model_lgbm,
    'CatBoost':    model_cat,
}

best_name  = results.loc[results['RMSE'].idxmin(), 'Model']
best_model = model_registry[best_name]
print(f"Best model by Test RMSE: {best_name}")
estimator = best_model.named_steps[list(best_model.named_steps.keys())[-1]]

#SHAP values
explainer   = shap.TreeExplainer(estimator)
shap_values = explainer.shap_values(X_test)
shap_exp    = explainer(X_test)   

#Plot 1: Summary beeswarm
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, show=False)
plt.title(f'SHAP Summary — {best_name}', fontsize=17, fontweight='bold')
plt.tight_layout()
plt.savefig('./resultados_CE/shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

#Plot 2: Feature importance bar
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False)
plt.title(f'SHAP Feature Importance — {best_name}', fontsize=17, fontweight='bold')
plt.tight_layout()
plt.savefig('./resultados_CE/shap_importance.png', dpi=300, bbox_inches='tight')
plt.show()
#Plot 3: Dependence plots
mean_shap  = np.abs(shap_values).mean(axis=0)
top5_idx   = np.argsort(mean_shap)[::-1][:5]
top5_feats = X_test.columns[top5_idx].tolist()

for feat in top5_feats:
    fig, ax = plt.subplots(figsize=(8, 6))
    shap.dependence_plot(feat, shap_values, X_test, ax=ax, show=False)
    ax.set_title(f'SHAP Dependence — {feat}  |  {best_name}', fontweight='bold')
    plt.tight_layout()
    slug_feat = feat.lower().replace(" ", "_")
    plt.savefig(f'./resultados_CE/shap_dependence_{slug_feat}.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams.update({
    'font.family':         'serif',
    'font.serif':          ['Times New Roman', 'DejaVu Serif'],
    'font.size':           16,
    'axes.titlesize':      18, 'axes.labelsize':      16,
    'xtick.labelsize':     16, 'ytick.labelsize':     16,
    'legend.fontsize':     16,
    'axes.linewidth':      1.2,
    'xtick.direction':     'in', 'ytick.direction':     'in',
    'xtick.major.size':    5, 'ytick.major.size':    5,
    'xtick.minor.visible': True, 'ytick.minor.visible': True,
    'xtick.minor.size':    2.5, 'ytick.minor.size':    2.5,
})

def ce_to_lce(ce_percent):
    return -np.log10(1 - ce_percent / 100)

def lce_to_ce(lce):
    return (1 - 10**(-lce)) * 100
kim_mse   = 0.343
best_name= results.loc[results['RMSE'].idxmin(), 'Model']
best_mse = results.loc[results['RMSE'].idxmin(), 'MSE']
kim_rmse  = np.sqrt(kim_mse)
best_rmse= np.sqrt(best_mse)
x_vals = np.linspace(80, 99.99, 500)
y_vals = ce_to_lce(x_vals)
puntos_ce = np.array([90, 95, 98, 99])
puntos_lce = ce_to_lce(puntos_ce)
COLOR_KIM= '#CC3333'
COLOR_BEST = '#1A1A1A'

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(x_vals, y_vals, color='#AAAAAA', linewidth=1.2, linestyle='--', label='LCE Transformation', zorder=1)
#Kim et al.(2023)
ax.errorbar(puntos_ce - 0.25, puntos_lce, yerr=kim_rmse, fmt='o', color=COLOR_KIM, capsize=4, capthick=1.2, markersize=6, markerfacecolor='white', markeredgewidth=1.5, linewidth=1.2, label=f'Kim et al. (2023) — MSE: {kim_mse:.4f}', zorder=3)
#Best model
ax.errorbar(puntos_ce + 0.25, puntos_lce, yerr=best_rmse, fmt='s', color=COLOR_BEST, capsize=4, capthick=1.2, markersize=6, markerfacecolor='white', markeredgewidth=1.5, linewidth=1.2, label=f'This work: {best_name} — MSE: {best_mse:.4f}', zorder=3)

ax.set_xlabel('Real Coulombic Efficiency (%)')
ax.set_ylabel('Logarithmic Coulombic Efficiency (LCE)')
ax.set_xlim(79, 100.3)
ax.set_ylim(0.5, ce_to_lce(99.99))
secax = ax.secondary_yaxis('right',functions=(lce_to_ce,lambda ce: -np.log10(1 - ce / 100)))
secax.set_ylabel('Coulombic Efficiency (%)', labelpad=10)
secax.tick_params(direction='in', which='both')
ce_ticks = [80, 85, 90, 95, 97, 98, 99, 99.5, 99.9, 99.99]
secax.set_yticks(ce_ticks)
secax.set_yticklabels([str(t) for t in ce_ticks])
ax.legend(frameon=True, framealpha=0.95, edgecolor='#CCCCCC', loc='upper left')
ax.set_title(f'Error Sensitivity: {best_name} vs. Kim et al. (2023)',fontweight='bold', pad=10)
ax.grid(True, linestyle=':', linewidth=0.6, alpha=0.6, color='#BBBBBB')
plt.tight_layout()
plt.savefig('./resultados_CE/error_sensitivity_comparison.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()